In [1]:
# ============================================================
# 🍽️ RESTAURANT FOOD PREPARATION PREDICTOR
# NLP + TF-IDF + XGBOOST + GRADIO
# ============================================================

!pip install -q xgboost gradio scikit-learn pandas numpy scipy


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import re
import warnings
import numpy as np
import pandas as pd
import gradio as gr

warnings.filterwarnings("ignore")

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import (
    train_test_split,
    KFold
)

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.feature_extraction.text import (
    TfidfVectorizer
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor


# ============================================================
# 2. LOAD YOUR REAL DATASET
# ============================================================

FILE_PATH = "restaurant_food_waste_dataset_100000.csv"

df = pd.read_csv(FILE_PATH)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Dish_Name",
    "Category",
    "Description",
    "Day_of_Week",
    "Month",
    "Season",
    "Weather",
    "Special_Event",
    "Price",
    "Prep_Time_Minutes",
    "Previous_Orders",
    "Ingredients_Stock",
    "Quantity_Prepared",
    "Orders_Placed",
    "Waste_Quantity"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:

    raise ValueError(
        "Missing columns in CSV: "
        + str(missing_columns)
    )

print("\nAll required columns are present.")


# ============================================================
# 4. REMOVE DUPLICATES
# ============================================================

before = len(df)

df = df.drop_duplicates()

after = len(df)

print(
    "\nDuplicates removed:",
    before - after
)


# ============================================================
# 5. HANDLE MISSING VALUES
# ============================================================

text_columns = [
    "Dish_Name",
    "Category",
    "Description",
    "Day_of_Week",
    "Season",
    "Weather",
    "Special_Event"
]

numeric_columns = [
    "Month",
    "Price",
    "Prep_Time_Minutes",
    "Previous_Orders",
    "Ingredients_Stock",
    "Quantity_Prepared",
    "Orders_Placed",
    "Waste_Quantity"
]


for col in text_columns:

    df[col] = (
        df[col]
        .fillna("Unknown")
        .astype(str)
    )


for col in numeric_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    df[col] = df[col].fillna(
        df[col].median()
    )


# ============================================================
# 6. NLP TEXT CLEANING
# ============================================================

print("\n" + "=" * 60)
print("NLP PROCESSING")
print("=" * 60)


def clean_text(text):

    text = str(text).lower()

    # Remove punctuation

    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        " ",
        text
    )

    # Remove extra spaces

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# Combine dish name + description

df["Food_Text"] = (
    df["Dish_Name"].astype(str)
    + " "
    + df["Description"].astype(str)
)


# Clean combined text

df["Food_Text"] = (
    df["Food_Text"]
    .apply(clean_text)
)


print("NLP text created.")

print("\nExample:")
print(df["Food_Text"].iloc[0])


# ============================================================
# 7. NLP: TF-IDF
# ============================================================

print("\nCreating TF-IDF features...")


tfidf = TfidfVectorizer(

    max_features=1000,

    ngram_range=(1, 2),

    min_df=2,

    max_df=0.95,

    sublinear_tf=True,

    stop_words="english"
)


text_matrix = tfidf.fit_transform(
    df["Food_Text"]
)


print(
    "TF-IDF Matrix Shape:",
    text_matrix.shape
)


print(
    "Vocabulary Size:",
    len(tfidf.vocabulary_)
)


# ============================================================
# 8. ENCODE CATEGORICAL FEATURES
# ============================================================

print("\nEncoding categorical features...")


categorical_columns = [
    "Category",
    "Day_of_Week",
    "Season",
    "Weather",
    "Special_Event"
]


encoder = OneHotEncoder(
    handle_unknown="ignore"
)


categorical_matrix = encoder.fit_transform(
    df[categorical_columns]
)


print(
    "Categorical Matrix Shape:",
    categorical_matrix.shape
)


# ============================================================
# 9. NUMERICAL FEATURES
# ============================================================

numeric_feature_names = [
    "Month",
    "Price",
    "Prep_Time_Minutes",
    "Previous_Orders",
    "Ingredients_Stock"
]


numeric_data = df[
    numeric_feature_names
].values


# ============================================================
# 10. FEATURE ENGINEERING
# ============================================================

# Previous order trend

previous_orders_log = np.log1p(
    df["Previous_Orders"].values
).reshape(-1, 1)


# Ingredient stock log

stock_log = np.log1p(
    df["Ingredients_Stock"].values
).reshape(-1, 1)


# Weekend feature

weekend = (
    df["Day_of_Week"]
    .isin(
        [
            "Saturday",
            "Sunday"
        ]
    )
    .astype(int)
    .values
    .reshape(-1, 1)
)


# Convert numerical features to float

numeric_data = numeric_data.astype(float)


# Scale numerical features

scaler = StandardScaler()

numeric_scaled = scaler.fit_transform(
    numeric_data
)


# Combine engineered numerical features

numeric_matrix = np.hstack(
    [
        numeric_scaled,
        previous_orders_log,
        stock_log,
        weekend
    ]
)


numeric_matrix = csr_matrix(
    numeric_matrix
)


print(
    "Numerical Matrix Shape:",
    numeric_matrix.shape
)


# ============================================================
# 11. COMBINE NLP + CATEGORICAL + NUMERICAL
# ============================================================

print("\nCombining all ML features...")


X_all = hstack(
    [
        text_matrix,
        categorical_matrix,
        numeric_matrix
    ],
    format="csr"
)


# Target

y_all = df[
    "Quantity_Prepared"
].values


print(
    "Final Feature Matrix:",
    X_all.shape
)

print(
    "Target Shape:",
    y_all.shape
)


# ============================================================
# 12. TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X_all,

    y_all,

    test_size=0.20,

    random_state=42
)


print("\n" + "=" * 60)
print("TRAIN TEST SPLIT")
print("=" * 60)

print(
    "Training:",
    X_train.shape
)

print(
    "Testing:",
    X_test.shape
)


# ============================================================
# 13. XGBOOST MODEL
# ============================================================

print("\n" + "=" * 60)
print("XGBOOST TRAINING")
print("=" * 60)


model = XGBRegressor(

    objective="reg:squarederror",

    n_estimators=300,

    learning_rate=0.05,

    max_depth=6,

    min_child_weight=3,

    subsample=0.8,

    colsample_bytree=0.8,

    reg_alpha=0.1,

    reg_lambda=1.0,

    random_state=42,

    n_jobs=-1
)


# ============================================================
# 14. TRAIN
# ============================================================

model.fit(

    X_train,

    y_train,

    verbose=False
)


print("Training completed!")


# ============================================================
# 15. PREDICTION
# ============================================================

y_pred = model.predict(
    X_test
)


# No negative preparation quantity

y_pred = np.maximum(
    y_pred,
    0
)


# ============================================================
# 16. MODEL EVALUATION
# ============================================================

mae = mean_absolute_error(
    y_test,
    y_pred
)


rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)


r2 = r2_score(
    y_test,
    y_pred
)


print("\n" + "=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)

print(
    "Algorithm: XGBoost Regressor"
)

print(
    "NLP: TF-IDF"
)

print(
    "MAE:",
    round(mae, 2)
)

print(
    "RMSE:",
    round(rmse, 2)
)

print(
    "R²:",
    round(r2, 4)
)


# ============================================================
# 17. CROSS VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("3-FOLD CROSS VALIDATION")
print("=" * 60)


# Use a sample to keep CV practical
# with 100,000 records.

cv_size = min(
    20000,
    X_train.shape[0]
)


rng = np.random.RandomState(42)

cv_indices = rng.choice(
    X_train.shape[0],
    size=cv_size,
    replace=False
)


X_cv = X_train[
    cv_indices
]

y_cv = y_train[
    cv_indices
]


kf = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


cv_scores = []


for fold, (train_idx, val_idx) in enumerate(
    kf.split(X_cv),
    start=1
):

    cv_model = XGBRegressor(

        objective="reg:squarederror",

        n_estimators=150,

        learning_rate=0.05,

        max_depth=6,

        subsample=0.8,

        colsample_bytree=0.8,

        random_state=42,

        n_jobs=-1
    )


    cv_model.fit(
        X_cv[train_idx],
        y_cv[train_idx],
        verbose=False
    )


    fold_prediction = cv_model.predict(
        X_cv[val_idx]
    )


    fold_prediction = np.maximum(
        fold_prediction,
        0
    )


    fold_r2 = r2_score(
        y_cv[val_idx],
        fold_prediction
    )


    cv_scores.append(
        fold_r2
    )


    print(
        f"Fold {fold} R²:",
        round(fold_r2, 4)
    )


print(
    "\nAverage CV R²:",
    round(
        np.mean(cv_scores),
        4
    )
)


# ============================================================
# 18. SIMPLE HYPERPARAMETER TUNING
# ============================================================

print("\n" + "=" * 60)
print("HYPERPARAMETER TUNING")
print("=" * 60)


# Small validation sample
# to avoid extremely long notebook execution.

tune_size = min(
    15000,
    X_train.shape[0]
)


tune_indices = rng.choice(
    X_train.shape[0],
    size=tune_size,
    replace=False
)


X_tune = X_train[
    tune_indices
]

y_tune = y_train[
    tune_indices
]


parameter_sets = [

    {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 5
    },

    {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 6
    },

    {
        "n_estimators": 300,
        "learning_rate": 0.1,
        "max_depth": 6
    },

    {
        "n_estimators": 400,
        "learning_rate": 0.05,
        "max_depth": 7
    }

]


best_params = None

best_mae = float("inf")


for i, params in enumerate(
    parameter_sets,
    start=1
):

    test_model = XGBRegressor(

        objective="reg:squarederror",

        n_estimators=params[
            "n_estimators"
        ],

        learning_rate=params[
            "learning_rate"
        ],

        max_depth=params[
            "max_depth"
        ],

        subsample=0.8,

        colsample_bytree=0.8,

        random_state=42,

        n_jobs=-1
    )


    test_model.fit(
        X_tune,
        y_tune,
        verbose=False
    )


    tune_prediction = (
        test_model.predict(
            X_test
        )
    )


    tune_prediction = np.maximum(
        tune_prediction,
        0
    )


    tune_mae = mean_absolute_error(
        y_test,
        tune_prediction
    )


    print(
        f"Configuration {i}:",
        params,
        "MAE =",
        round(tune_mae, 2)
    )


    if tune_mae < best_mae:

        best_mae = tune_mae

        best_params = params


print(
    "\nBest Parameters:"
)

print(
    best_params
)


# ============================================================
# 19. TRAIN FINAL MODEL
# ============================================================

print("\n" + "=" * 60)
print("TRAINING FINAL MODEL")
print("=" * 60)


final_model = XGBRegressor(

    objective="reg:squarederror",

    n_estimators=best_params[
        "n_estimators"
    ],

    learning_rate=best_params[
        "learning_rate"
    ],

    max_depth=best_params[
        "max_depth"
    ],

    min_child_weight=3,

    subsample=0.8,

    colsample_bytree=0.8,

    reg_alpha=0.1,

    reg_lambda=1.0,

    random_state=42,

    n_jobs=-1
)


final_model.fit(

    X_train,

    y_train,

    verbose=False
)


# ============================================================
# 20. FINAL EVALUATION
# ============================================================

final_prediction = final_model.predict(
    X_test
)


final_prediction = np.maximum(
    final_prediction,
    0
)


final_mae = mean_absolute_error(
    y_test,
    final_prediction
)


final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        final_prediction
    )
)


final_r2 = r2_score(
    y_test,
    final_prediction
)


print("\n" + "=" * 60)
print("FINAL MODEL")
print("=" * 60)


print(
    "Final MAE:",
    round(final_mae, 2)
)


print(
    "Final RMSE:",
    round(final_rmse, 2)
)


print(
    "Final R²:",
    round(final_r2, 4)
)


# ============================================================
# 21. GRADIO DATA VALUES
# ============================================================

dish_choices = sorted(
    df["Dish_Name"]
    .unique()
    .tolist()
)


category_choices = sorted(
    df["Category"]
    .unique()
    .tolist()
)


season_choices = sorted(
    df["Season"]
    .unique()
    .tolist()
)


weather_choices = sorted(
    df["Weather"]
    .unique()
    .tolist()
)


event_choices = sorted(
    df["Special_Event"]
    .unique()
    .tolist()
)


day_choices = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]


# ============================================================
# 22. GRADIO PREDICTION FUNCTION
# ============================================================

def predict_food(

    dish_name,

    description,

    category,

    day_of_week,

    month,

    season,

    weather,

    special_event,

    price,

    prep_time,

    previous_orders,

    ingredients_stock

):

    try:

        # ----------------------------------------------
        # TEXT / NLP
        # ----------------------------------------------

        combined_text = (
            str(dish_name)
            + " "
            + str(description)
        )


        combined_text = clean_text(
            combined_text
        )


        # TF-IDF transform
        # IMPORTANT: transform, NOT fit_transform

        text_features = tfidf.transform(
            [combined_text]
        )


        # ----------------------------------------------
        # CATEGORICAL
        # ----------------------------------------------

        categorical_input = pd.DataFrame({

            "Category": [
                category
            ],

            "Day_of_Week": [
                day_of_week
            ],

            "Season": [
                season
            ],

            "Weather": [
                weather
            ],

            "Special_Event": [
                special_event
            ]

        })


        categorical_features_input = (
            encoder.transform(
                categorical_input
            )
        )


        # ----------------------------------------------
        # NUMERICAL
        # ----------------------------------------------

        day_number_map = {

            "Monday": 0,

            "Tuesday": 1,

            "Wednesday": 2,

            "Thursday": 3,

            "Friday": 4,

            "Saturday": 5,

            "Sunday": 6

        }


        day_number = day_number_map.get(
            day_of_week,
            0
        )


        is_weekend = int(
            day_number >= 5
        )


        numeric_input = np.array([

            float(month),

            float(price),

            float(prep_time),

            float(previous_orders),

            float(ingredients_stock)

        ]).reshape(
            1,
            -1
        )


        numeric_scaled_input = (
            scaler.transform(
                numeric_input
            )
        )


        previous_orders_log = np.array([

            np.log1p(
                float(previous_orders)
            )

        ]).reshape(
            1,
            -1
        )


        stock_log = np.array([

            np.log1p(
                float(ingredients_stock)
            )

        ]).reshape(
            1,
            -1
        )


        weekend_input = np.array([

            is_weekend

        ]).reshape(
            1,
            -1
        )


        numerical_final = np.hstack([

            numeric_scaled_input,

            previous_orders_log,

            stock_log,

            weekend_input

        ])


        numerical_final = csr_matrix(
            numerical_final
        )


        # ----------------------------------------------
        # COMBINE ALL FEATURES
        # ----------------------------------------------

        final_input = hstack(

            [

                text_features,

                categorical_features_input,

                numerical_final

            ],

            format="csr"

        )


        # ----------------------------------------------
        # PREDICT
        # ----------------------------------------------

        prediction = final_model.predict(
            final_input
        )[0]


        prediction = max(
            0,
            prediction
        )


        recommended_quantity = int(
            np.ceil(
                prediction
            )
        )


        return (

            f"🍽️ Predicted Preparation Quantity\n\n"

            f"➡️ {recommended_quantity} portions\n\n"

            f"📊 Expected Demand: "
            f"{prediction:.0f} portions\n\n"

            f"🧠 NLP Input: "
            f"{combined_text}\n\n"

            f"🤖 Model: XGBoost + TF-IDF\n\n"

            f"♻️ Use this prediction to support "
            f"daily food preparation planning."
        )


    except Exception as e:

        return (
            "❌ Prediction Error:\n\n"
            + str(e)
        )


# ============================================================
# 23. GRADIO INTERFACE
# ============================================================

with gr.Blocks(

    title="Restaurant Food Waste Predictor"

) as demo:


    gr.Markdown(

        """
        # 🍽️ Restaurant Food Preparation Predictor

        ## 🧠 NLP + Machine Learning

        This system predicts how many portions of a dish
        should be prepared.

        **NLP:** Dish Name + Description → TF-IDF

        **ML:** XGBoost Regression

        **Goal:** Better preparation planning and reduced
        unnecessary food waste.
        """

    )


    with gr.Row():


        with gr.Column():


            dish_name = gr.Dropdown(

                choices=dish_choices,

                value=dish_choices[0],

                label="🍴 Dish Name"

            )


            description = gr.Textbox(

                label="📝 Dish Description",

                value=str(
                    df[
                        df["Dish_Name"]
                        == dish_choices[0]
                    ]["Description"].iloc[0]
                )

            )


            category = gr.Dropdown(

                choices=category_choices,

                value=category_choices[0],

                label="🍱 Category"

            )


            day_of_week = gr.Dropdown(

                choices=day_choices,

                value="Saturday",

                label="📅 Day of Week"

            )


            month = gr.Slider(

                minimum=1,

                maximum=12,

                step=1,

                value=12,

                label="📆 Month"

            )


            season = gr.Dropdown(

                choices=season_choices,

                value=season_choices[0],

                label="🌤️ Season"

            )


        with gr.Column():


            weather = gr.Dropdown(

                choices=weather_choices,

                value=weather_choices[0],

                label="🌦️ Weather"

            )


            special_event = gr.Dropdown(

                choices=event_choices,

                value=event_choices[0],

                label="🎉 Special Event"

            )


            price = gr.Number(

                label="💰 Price",

                value=200

            )


            prep_time = gr.Number(

                label="⏱️ Preparation Time",

                value=20

            )


            previous_orders = gr.Number(

                label="📊 Previous Orders",

                value=100

            )


            ingredients_stock = gr.Number(

                label="🥬 Ingredients Stock",

                value=100

            )


    # ========================================================
    # BUTTON
    # ========================================================

    predict_button = gr.Button(

        "🔮 Predict Food Preparation Quantity",

        variant="primary"

    )


    # ========================================================
    # OUTPUT
    # ========================================================

    output = gr.Textbox(

        label="Prediction Result",

        lines=8

    )


    predict_button.click(

        fn=predict_food,

        inputs=[

            dish_name,

            description,

            category,

            day_of_week,

            month,

            season,

            weather,

            special_event,

            price,

            prep_time,

            previous_orders,

            ingredients_stock

        ],

        outputs=output

    )


    # ========================================================
    # PROJECT INFORMATION
    # ========================================================

    gr.Markdown(

        f"""
        ---

        ## 📊 Model Performance

        **Dataset:** 100,000 Restaurant Records

        **Target:** Quantity Prepared

        **NLP:** TF-IDF

        **ML Model:** XGBoost Regressor

        **Train/Test:** 80% / 20%

        **Cross-Validation:** 3-Fold

        **Hyperparameter Tuning:** Enabled

        **MAE:** {final_mae:.2f}

        **RMSE:** {final_rmse:.2f}

        **R² Score:** {final_r2:.4f}

        ---

        ## 🧠 NLP Pipeline

        Dish Name

        ↓

        Dish Description

        ↓

        Text Cleaning

        ↓

        Tokenization / Text Representation

        ↓

        TF-IDF

        ↓

        Numerical NLP Features

        ↓

        XGBoost

        ↓

        Predicted Preparation Quantity

        ---

        ## 🔬 Machine Learning Techniques

        ✅ Data Cleaning

        ✅ NLP Text Preprocessing

        ✅ TF-IDF

        ✅ One-Hot Encoding

        ✅ Feature Engineering

        ✅ Feature Scaling

        ✅ Train-Test Split

        ✅ XGBoost Regression

        ✅ Cross-Validation

        ✅ Hyperparameter Tuning

        ✅ MAE

        ✅ RMSE

        ✅ R²

        ---

        ## ♻️ Real-World Impact

        The system can help restaurants make more
        data-driven food preparation decisions.

        Better demand estimation can reduce unnecessary
        preparation and support inventory planning.

        ---

        ## 🚀 Future Improvements

        • BERT / Sentence Transformers

        • Customer reviews and sentiment

        • Holiday information

        • Customer footfall

        • Real-time orders

        • Restaurant location

        • Real-time weather

        • Actual food-waste monitoring

        • Mobile / web deployment
        """

    )


# ============================================================
# 24. LAUNCH
# ============================================================

print("\n" + "=" * 60)
print("🚀 GRADIO APPLICATION")
print("=" * 60)

demo.launch(
    share=True
)

DATASET LOADED
Shape: (100000, 17)

Columns:
['Record_ID', 'Dish_ID', 'Dish_Name', 'Category', 'Description', 'Day_of_Week', 'Month', 'Season', 'Weather', 'Special_Event', 'Price', 'Prep_Time_Minutes', 'Previous_Orders', 'Ingredients_Stock', 'Quantity_Prepared', 'Orders_Placed', 'Waste_Quantity']

First 5 rows:
   Record_ID Dish_ID          Dish_Name  Category  \
0          1    D004        Butter Naan     Bread   
1          2    D031            Chapati     Bread   
2          3    D027  Chocolate Brownie   Dessert   
3          4    D018        Mango Lassi  Beverage   
4          5    D018        Mango Lassi  Beverage   

                           Description Day_of_Week  Month   Season Weather  \
0  traditional tandoor baked flatbread   Wednesday     12   Winter  Cloudy   
1  traditional tandoor baked flatbread    Saturday     12   Winter   Sunny   
2               rich creamy sweet dish      Sunday     12   Winter     Hot   
3    hot beverage with milk and spices    Saturday     1

2026/10/06 12:32:18 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout
